# Dijet generator-smearing effect

Compare nominal generator, generator-level smearing, eta-dependent generator-level smearing, and reconstructed dijet pseudorapidity distributions for one configured jet $\eta_{CM}$ acceptance and every interval in `DIJET_PTAVE_BINS`. Raw full $\eta_{CM}^{dijet}$ distributions and their ratios are drawn before normalization, followed by the normalized full-shape comparisons. Forward and backward projections remain unnormalized when their ratio is formed. Overlays, ratios to nominal Gen, and ratios to eta-dependent smeared Gen are drawn on separate canvases.

In [ ]:
%load_ext autoreload
%autoreload 2

from pathlib import Path
import math
import os
import sys

PROJECT_ROOT = next(
    (path for path in (Path.cwd(), *Path.cwd().parents)
     if (path / 'hist_analysis').is_dir()),
    None,
)
if PROJECT_ROOT is None:
    raise RuntimeError('Run this notebook from the jetAnalysis repository root')
if str(PROJECT_ROOT) not in sys.path:
    sys.path.insert(0, str(PROJECT_ROOT))

try:
    import ROOT
except ModuleNotFoundError:
    for path in (Path('/opt/homebrew/lib/python3.14/site-packages'),
                 Path('/opt/homebrew/Cellar/root/6.40.02_1/lib/root')):
        if path.exists() and str(path) not in sys.path:
            sys.path.insert(0, str(path))
    import ROOT

ROOT.gROOT.SetBatch(True)
ROOT.gStyle.SetOptStat(0)
ROOT.TH1.AddDirectory(False)
ROOT.gStyle.SetPalette(ROOT.kBird)

from hist_analysis.config.files import BASE_DIR
from hist_analysis.config.histograms import (
    DIJET_DELTA_PHI_SELECTION_LABEL, DIJET_PTAVE_BINS,
    STANDARD_DIJET_ETA_CUT_INDEX,
)
from hist_analysis.python.dijet_closures import (
    DijetClosureCurve, build_dijet_gen_comparisons,
)
from hist_analysis.python.histogram_io import (
    resolve_combined_file, resolve_direction_file,
)
from hist_analysis.python.histogram_ops import ratio_to_nominal
from hist_analysis.python.plotting import draw_overlay

## Configuration

`CURVE_CATALOG` defines every available distribution and its keys, style, raw scaling, and Reco-like error treatment. `REQUIRED_CURVE_LABELS` always keeps nominal Reco, nominal Gen, and eta-dependent smeared Gen. Enable or disable any other catalog entry by editing only `OPTIONAL_CURVE_LABELS`. Every histogram key template uses `{eta_cut_index}` to select the stored acceptance. `ETA_CUT_INDEX=5` corresponds to $|\eta_{CM}^{jet}|<1.9$, and projection intervals are half-open. Generator-smeared histograms accumulate `N_SMEAR_RUNS=20` trials per event, while eta-dependent smeared Reco accumulates `N_RECO_SMEAR_RUNS=10`; only their raw projections are scaled by the corresponding inverse run count. Generator-level full-distribution ratios use ROOT binomial uncertainty propagation, while Reco-to-generator-reference full ratios use `RECO_FULL_COMPARISON_RATIO_OPTION`. Each `F/B` is always constructed with independent propagation; the later ratio of two F/B histograms is controlled separately by `FB_COMPARISON_RATIO_OPTION`.

In [ ]:
GENERATOR = 'embedding'       # embedding or pythia
DIRECTION = 'combined'        # pgoing, Pbgoing, or combined
FILE_STEM = 'jetId'
ETA_CUTS = (1.4, 1.5, 1.6, 1.7, 1.8, 1.9, 2.5)
ETA_CUT_INDEX = STANDARD_DIJET_ETA_CUT_INDEX
PTAVE_BINS = tuple(DIJET_PTAVE_BINS)
REBIN_ETA = 2
N_SMEAR_RUNS = 20            # trials accumulated in Gen-smeared histograms
N_RECO_SMEAR_RUNS = 10       # trials accumulated in Reco-smeared histograms
NORMALIZATION = 'integral'   # required for 1/N dN/deta density overlays
FORWARD_BACKWARD_RATIO_OPTION = ''  # F / B: never use binomial errors
FULL_COMPARISON_RATIO_OPTION = 'B'   # generator-level full-distribution ratios
RECO_FULL_COMPARISON_RATIO_OPTION = 'B'  # Reco / generator reference
FB_COMPARISON_RATIO_OPTION = 'B'      # user choice for (F/B)_a / (F/B)_b: '' or 'B'
FULL_RATIO_RANGE = (0.85, 1.3)
RAW_FULL_RATIO_TO_GEN_RANGE = (0.75, 1.3)
RAW_FULL_RATIO_TO_ETA_DEPENDENT_RANGE = (0.75, 1.3)
FB_RANGE = (0.9, 1.30)
FB_RATIO_RANGE = (0.9, 1.3)
DRAW_GRID = True
SAVE_PNG = False
OUTPUT_DIR = Path(os.environ.get(
    'DIJET_SMEARING_EFFECT_OUTPUT_DIR',
    PROJECT_ROOT / 'hist_analysis' / 'output' / 'dijet_smearing_effect',
))

CURVE_CATALOG = {
    'Gen': {
        'curve': DijetClosureCurve(
            'Gen', 'hGenDijetPtEtaCM_{eta_cut_index}',
            'hGenDijetPtEtaForward_{eta_cut_index}',
            'hGenDijetPtEtaBackward_{eta_cut_index}',
        ),
        'style_index': 2, 'raw_scale': 1.0, 'reco_like': False,
    },
    'Gen JER (#eta-dep.)': {
        'curve': DijetClosureCurve(
            'Gen JER (#eta-dep.)',
            'hGenDijetDefExtraPtEtaCM_{eta_cut_index}',
            'hGenDijetDefExtraPtEtaForward_{eta_cut_index}',
            'hGenDijetDefExtraPtEtaBackward_{eta_cut_index}',
        ),
        'style_index': 1, 'raw_scale': 1.0 / N_SMEAR_RUNS,
        'reco_like': False,
    },
    'Reco': {
        'curve': DijetClosureCurve(
            'Reco', 'hRecoDijetPtEtaCM_{eta_cut_index}',
            'hRecoDijetPtEtaForward_{eta_cut_index}',
            'hRecoDijetPtEtaBackward_{eta_cut_index}',
        ),
        'style_index': 0, 'raw_scale': 1.0, 'reco_like': True,
    },
    'Gen JER (x 1.0)': {
        'curve': DijetClosureCurve(
            'Gen JER (x 1.0)', 'hGenDijetDefPtEtaCM_{eta_cut_index}',
            'hGenDijetDefPtEtaForward_{eta_cut_index}',
            'hGenDijetDefPtEtaBackward_{eta_cut_index}',
        ),
        'style_index': 3, 'raw_scale': 1.0 / N_SMEAR_RUNS,
        'reco_like': False,
    },
    'Reco JER (def.+#eta-dep.)': {
        'curve': DijetClosureCurve(
            'Reco JER (def.+#eta-dep.)',
            'hRecoDijetPtEtaCMJerDefExtra_{eta_cut_index}',
            'hRecoDijetPtEtaForwardJerDefExtra_{eta_cut_index}',
            'hRecoDijetPtEtaBackwardJerDefExtra_{eta_cut_index}',
        ),
        'style_index': 5, 'raw_scale': 1.0 / N_RECO_SMEAR_RUNS,
        'reco_like': True,
    },
}

# These three curves are always present.
REQUIRED_CURVE_LABELS = ('Gen', 'Gen JER (#eta-dep.)', 'Reco')
# Add or remove catalog labels here; no plotting code changes are needed.
OPTIONAL_CURVE_LABELS = (
    # 'Gen JER (x 1.0)',
    'Reco JER (def.+#eta-dep.)',
)
ACTIVE_CURVE_LABELS = REQUIRED_CURVE_LABELS + OPTIONAL_CURVE_LABELS
NOMINAL = 'Gen'
ETA_DEPENDENT_NOMINAL = 'Gen JER (#eta-dep.)'

if ETA_CUT_INDEX < 0 or ETA_CUT_INDEX >= len(ETA_CUTS):
    raise IndexError(f'Invalid eta-cut index: {ETA_CUT_INDEX}')
if isinstance(N_SMEAR_RUNS, bool) or not isinstance(N_SMEAR_RUNS, int) or N_SMEAR_RUNS < 1:
    raise ValueError(f'N_SMEAR_RUNS must be a positive integer, got {N_SMEAR_RUNS!r}')
if (isinstance(N_RECO_SMEAR_RUNS, bool)
        or not isinstance(N_RECO_SMEAR_RUNS, int) or N_RECO_SMEAR_RUNS < 1):
    raise ValueError(
        f'N_RECO_SMEAR_RUNS must be a positive integer, got {N_RECO_SMEAR_RUNS!r}'
    )
if len(set(ACTIVE_CURVE_LABELS)) != len(ACTIVE_CURVE_LABELS):
    raise ValueError(f'Active curve labels must be unique: {ACTIVE_CURVE_LABELS}')
unknown_curve_labels = set(ACTIVE_CURVE_LABELS) - set(CURVE_CATALOG)
if unknown_curve_labels:
    raise ValueError(f'Unknown active curves: {sorted(unknown_curve_labels)}')
CURVES = tuple(CURVE_CATALOG[label]['curve'] for label in ACTIVE_CURVE_LABELS)
STYLE_INDICES = {
    label: CURVE_CATALOG[label]['style_index'] for label in ACTIVE_CURVE_LABELS
}
RAW_SCALE_FACTORS = {
    label: CURVE_CATALOG[label]['raw_scale'] for label in ACTIVE_CURVE_LABELS
    if CURVE_CATALOG[label]['raw_scale'] != 1.0
}
RECO_CURVE_LABELS = {
    label for label in ACTIVE_CURVE_LABELS if CURVE_CATALOG[label]['reco_like']
}
ETA_CUT = ETA_CUTS[ETA_CUT_INDEX]

## Resolve the input

Combined and direction-specific samples use the common repository file resolvers. Histogram existence, TH2 type, and common projected eta binning are validated while each pTave interval is built.

In [ ]:
def mc_file(generator, direction):
    if direction == 'combined':
        return resolve_combined_file(BASE_DIR, generator, FILE_STEM)
    return resolve_direction_file(BASE_DIR, generator, direction, FILE_STEM)

DIRECTION_LABELS = {
    'pgoing': 'p-going',
    'Pbgoing': 'Pb-going',
    'combined': 'combined',
}
if DIRECTION not in DIRECTION_LABELS:
    raise ValueError(f'Unsupported DIRECTION={DIRECTION!r}')
DIRECTION_LABEL = DIRECTION_LABELS[DIRECTION]

INPUT_FILE = mc_file(GENERATOR, DIRECTION)
if not INPUT_FILE.exists():
    raise FileNotFoundError(f'Missing configured ROOT file: {INPUT_FILE}')
INPUT_FILE

## Full-shape and forward/backward comparisons

For each configured pTave interval, nine independent canvases are produced for every active curve. The first three show the raw full-distribution overlay and raw ratios to Gen and eta-dependent smeared Gen after applying each catalog entry's raw scale. They are followed by the corresponding three normalized full-shape canvases, the unnormalized forward/backward-ratio overlay, and the two F/B comparison ratios. Ratio histograms are never placed in a lower pad.

In [ ]:
comparison_results = {}
eta_x_range = (-ETA_CUT - 0.1, ETA_CUT + 0.1)
fb_x_range = (0.0, ETA_CUT + 0.1)
eta_cut_tag = int(round(10.0 * ETA_CUT))

for ptave_range in PTAVE_BINS:
    low, high = ptave_range
    ptave_tag = f'{low:g}_{high:g}'.replace('.', 'p')
    selection_tag = f'etaCM_{eta_cut_tag}_ptave_{ptave_tag}'
    common_tag = f'{GENERATOR}_{DIRECTION}_{selection_tag}'
    output_name = lambda plot: f'{GENERATOR}_{DIRECTION}_{plot}_{selection_tag}.pdf'
    raw_eta_shapes, _, raw_selected_keys = build_dijet_gen_comparisons(
        INPUT_FILE, CURVES, eta_cut_index=ETA_CUT_INDEX,
        ptave_range=ptave_range, nominal=NOMINAL, rebin_eta=REBIN_ETA,
        normalization='none',
        ratio_option=FORWARD_BACKWARD_RATIO_OPTION,
    )
    for label, scale_factor in RAW_SCALE_FACTORS.items():
        raw_eta_shapes[label].Scale(scale_factor)
    eta_shapes, fb_ratios, selected_keys = build_dijet_gen_comparisons(
        INPUT_FILE, CURVES, eta_cut_index=ETA_CUT_INDEX,
        ptave_range=ptave_range, nominal=NOMINAL, rebin_eta=REBIN_ETA,
        normalization=NORMALIZATION,
        ratio_option=FORWARD_BACKWARD_RATIO_OPTION,
    )
    raw_eta_to_gen = {
        label: ratio_to_nominal(
            histogram, raw_eta_shapes[NOMINAL],
            name=f'h_{common_tag}_{label.replace(" ", "_")}_raw_to_gen',
            option=(RECO_FULL_COMPARISON_RATIO_OPTION
                    if label in RECO_CURVE_LABELS else FULL_COMPARISON_RATIO_OPTION),
        )
        for label, histogram in raw_eta_shapes.items() if label != NOMINAL
    }
    raw_eta_to_eta_dependent = {
        label: ratio_to_nominal(
            histogram, raw_eta_shapes[ETA_DEPENDENT_NOMINAL],
            name=f'h_{common_tag}_{label.replace(" ", "_")}_raw_to_gen_eta_dep',
            option=(RECO_FULL_COMPARISON_RATIO_OPTION
                    if label in RECO_CURVE_LABELS else FULL_COMPARISON_RATIO_OPTION),
        )
        for label, histogram in raw_eta_shapes.items()
        if label != ETA_DEPENDENT_NOMINAL
    }
    eta_to_gen = {
        label: ratio_to_nominal(
            histogram, eta_shapes[NOMINAL],
            name=f'h_{common_tag}_{label.replace(" ", "_")}_to_gen',
            option=(RECO_FULL_COMPARISON_RATIO_OPTION
                    if label in RECO_CURVE_LABELS else FULL_COMPARISON_RATIO_OPTION),
        )
        for label, histogram in eta_shapes.items() if label != NOMINAL
    }
    fb_to_gen = {
        label: ratio_to_nominal(
            histogram, fb_ratios[NOMINAL],
            name=f'h_{common_tag}_{label.replace(" ", "_")}_fb_to_gen',
            option=FB_COMPARISON_RATIO_OPTION,
        )
        for label, histogram in fb_ratios.items() if label != NOMINAL
    }
    eta_to_eta_dependent = {
        label: ratio_to_nominal(
            histogram, eta_shapes[ETA_DEPENDENT_NOMINAL],
            name=f'h_{common_tag}_{label.replace(" ", "_")}_to_gen_eta_dep',
            option=(RECO_FULL_COMPARISON_RATIO_OPTION
                    if label in RECO_CURVE_LABELS else FULL_COMPARISON_RATIO_OPTION),
        )
        for label, histogram in eta_shapes.items()
        if label != ETA_DEPENDENT_NOMINAL
    }
    fb_to_eta_dependent = {
        label: ratio_to_nominal(
            histogram, fb_ratios[ETA_DEPENDENT_NOMINAL],
            name=f'h_{common_tag}_{label.replace(" ", "_")}_fb_to_gen_eta_dep',
            option=FB_COMPARISON_RATIO_OPTION,
        )
        for label, histogram in fb_ratios.items()
        if label != ETA_DEPENDENT_NOMINAL
    }
    annotations = (
        GENERATOR.capitalize(),
        DIRECTION_LABEL,
        'CM frame',
        f'{low:g} < p_{{T}}^{{ave}} < {high:g} GeV',
        f'|#eta_{{CM}}^{{jet}}| < {ETA_CUT:g}',
        'p_{T}^{Lead} > 50 GeV',
        'p_{T}^{SubLead} > 40 GeV',
        DIJET_DELTA_PHI_SELECTION_LABEL,
    )
    canvases = {
        'raw_eta_overlay': draw_overlay(
            raw_eta_shapes, title='', x_title='#eta_{CM}^{dijet}',
            y_title='dN/d#eta_{CM}^{dijet}', x_range=eta_x_range,
            annotations=annotations, grid=DRAW_GRID, headroom=1.55,
            style_indices=STYLE_INDICES,
            output=OUTPUT_DIR / output_name('full_raw_overlay'),
            save_png=SAVE_PNG, canvas_name=f'{common_tag}_full_raw_overlay',
        ),
        'raw_eta_ratio': draw_overlay(
            raw_eta_to_gen, title='', x_title='#eta_{CM}^{dijet}',
            y_title='Raw ratio to Gen', x_range=eta_x_range,
            y_range=RAW_FULL_RATIO_TO_GEN_RANGE, reference_y=1.0,
            annotations=annotations, grid=DRAW_GRID,
            style_indices=STYLE_INDICES,
            output=OUTPUT_DIR / output_name('full_raw_ratio_to_gen'),
            save_png=SAVE_PNG,
            canvas_name=f'{common_tag}_full_raw_ratio_to_gen',
        ),
        'raw_eta_ratio_to_eta_dependent': draw_overlay(
            raw_eta_to_eta_dependent, title='',
            x_title='#eta_{CM}^{dijet}',
            y_title='Raw ratio to Gen (JER #eta-dep.)',
            x_range=eta_x_range,
            y_range=RAW_FULL_RATIO_TO_ETA_DEPENDENT_RANGE,
            reference_y=1.0, annotations=annotations, grid=DRAW_GRID,
            style_indices=STYLE_INDICES,
            output=OUTPUT_DIR / output_name(
                'full_raw_ratio_to_gen_eta_dep'),
            save_png=SAVE_PNG,
            canvas_name=f'{common_tag}_full_raw_ratio_to_gen_eta_dep',
        ),
        'eta_overlay': draw_overlay(
            eta_shapes, title='', x_title='#eta_{CM}^{dijet}',
            y_title='1/N dN/d#eta_{CM}^{dijet}', x_range=eta_x_range,
            annotations=annotations, grid=DRAW_GRID, headroom=1.55,
            style_indices=STYLE_INDICES,
            output=OUTPUT_DIR / output_name('full_overlay'),
            save_png=SAVE_PNG, canvas_name=f'{common_tag}_full_overlay',
        ),
        'eta_ratio': draw_overlay(
            eta_to_gen, title='', x_title='#eta_{CM}^{dijet}',
            y_title='Ratio to Gen', x_range=eta_x_range,
            y_range=FULL_RATIO_RANGE, reference_y=1.0,
            annotations=annotations, grid=DRAW_GRID,
            style_indices=STYLE_INDICES,
            output=OUTPUT_DIR / output_name('full_ratio_to_gen'),
            save_png=SAVE_PNG, canvas_name=f'{common_tag}_full_ratio',
        ),
        'eta_ratio_to_eta_dependent': draw_overlay(
            eta_to_eta_dependent, title='', x_title='#eta_{CM}^{dijet}',
            y_title='Ratio to Gen (JER #eta-dep.)', x_range=eta_x_range,
            y_range=FULL_RATIO_RANGE, reference_y=1.0,
            annotations=annotations, grid=DRAW_GRID,
            style_indices=STYLE_INDICES,
            output=OUTPUT_DIR / output_name('full_ratio_to_gen_eta_dep'),
            save_png=SAVE_PNG,
            canvas_name=f'{common_tag}_full_ratio_to_gen_eta_dep',
        ),
        'fb_overlay': draw_overlay(
            fb_ratios, title='', x_title='#eta_{CM}^{dijet}',
            y_title='Forward / Backward', x_range=fb_x_range,
            y_range=FB_RANGE, reference_y=1.0, annotations=annotations,
            grid=DRAW_GRID, style_indices=STYLE_INDICES,
            output=OUTPUT_DIR / output_name('fb_overlay'),
            save_png=SAVE_PNG, canvas_name=f'{common_tag}_fb_overlay',
        ),
        'fb_ratio': draw_overlay(
            fb_to_gen, title='', x_title='#eta_{CM}^{dijet}',
            y_title='(Forward / Backward) ratio to Gen',
            x_range=fb_x_range, y_range=FB_RATIO_RANGE, reference_y=1.0,
            annotations=annotations, grid=DRAW_GRID,
            style_indices=STYLE_INDICES,
            output=OUTPUT_DIR / output_name('fb_ratio_to_gen'),
            save_png=SAVE_PNG, canvas_name=f'{common_tag}_fb_ratio',
        ),
        'fb_ratio_to_eta_dependent': draw_overlay(
            fb_to_eta_dependent, title='', x_title='#eta_{CM}^{dijet}',
            y_title='(Forward / Backward) ratio to Gen (JER #eta-dep.)',
            x_range=fb_x_range, y_range=FB_RATIO_RANGE, reference_y=1.0,
            annotations=annotations, grid=DRAW_GRID,
            style_indices=STYLE_INDICES,
            output=OUTPUT_DIR / output_name('fb_ratio_to_gen_eta_dep'),
            save_png=SAVE_PNG,
            canvas_name=f'{common_tag}_fb_ratio_to_gen_eta_dep',
        ),
    }
    comparison_results[ptave_range] = {
        'raw_eta_shapes': raw_eta_shapes,
        'raw_eta_to_gen': raw_eta_to_gen,
        'raw_eta_to_eta_dependent': raw_eta_to_eta_dependent,
        'eta_shapes': eta_shapes, 'eta_to_gen': eta_to_gen,
        'eta_to_eta_dependent': eta_to_eta_dependent,
        'forward_backward': fb_ratios, 'forward_backward_to_gen': fb_to_gen,
        'forward_backward_to_eta_dependent': fb_to_eta_dependent,
        'canvases': canvases, 'raw_keys': raw_selected_keys,
        'keys': selected_keys,
    }
    print(common_tag, selected_keys)
    for canvas in canvases.values():
        display(canvas)

## Numerical audit

Report the smear-run-corrected raw full-distribution yields, normalized full-shape integrals, and finite extrema of every raw and normalized comparison to Gen and eta-dependent smeared Gen. This exposes bins outside the configured display ranges without modifying their contents.

In [ ]:
def finite_nonzero_range(histogram):
    values = [
        histogram.GetBinContent(index)
        for index in range(1, histogram.GetNbinsX() + 1)
        if histogram.GetBinContent(index) != 0.0
        and math.isfinite(histogram.GetBinContent(index))
    ]
    return (min(values), max(values)) if values else None

for ptave_range, result in comparison_results.items():
    print(f'\npTave interval {ptave_range}, eta cut {ETA_CUT:g}')
    for label in (curve.label for curve in CURVES):
        print(
            f'  {label:27s} raw yield='
            f'{result["raw_eta_shapes"][label].Integral():.8g}, '
            f'full width integral='
            f'{result["eta_shapes"][label].Integral("width"):.8g}, '
            f'F/B range={finite_nonzero_range(result["forward_backward"][label])}'
        )
        if label != NOMINAL:
            print(
                f'    raw/Gen range='
                f'{finite_nonzero_range(result["raw_eta_to_gen"][label])}, '
                f'full/Gen range='
                f'{finite_nonzero_range(result["eta_to_gen"][label])}, '
                f'(F/B)/(F/B)_Gen range='
                f'{finite_nonzero_range(result["forward_backward_to_gen"][label])}'
            )
        if label != ETA_DEPENDENT_NOMINAL:
            print(
                f'    raw/Gen-eta-dep range='
                f'{finite_nonzero_range(result["raw_eta_to_eta_dependent"][label])}, '
                f'full/Gen-eta-dep range='
                f'{finite_nonzero_range(result["eta_to_eta_dependent"][label])}, '
                f'(F/B)/(F/B)_Gen-eta-dep range='
                f'{finite_nonzero_range(result["forward_backward_to_eta_dependent"][label])}'
            )
